In [2]:
import joblib
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


In [3]:
# Load dataset
df = pd.read_csv("../../datas/Iris.csv")

In [4]:
# Split the dataset into training and testing sets
train, test = train_test_split(df, test_size = 0.3)# in this our main data is split into train and test

# Just tocheck how many to train and test data or if I am in the right direction
print(train.shape)
print(test.shape)


(105, 6)
(45, 6)


In [5]:
df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [6]:

train_X = train[['SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm']]# taking the training data features
train_y=train.Species# output of our training data
test_X= test[['SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm']] # taking test data features
test_y =test.Species   #output value of test data

In [7]:
params = {
    "n_estimators": 100,
    "random_state": 45,
}

In [8]:
import mlflow

# Enable autologging for scikit-learn
mlflow.sklearn.autolog()


# Start an MLflow run
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Train the model
    RFC = RandomForestClassifier (**params)
    RFC.fit(train_X, train_y)

    # Log the model
    model_info = mlflow.sklearn.log_model(sk_model=RFC, name="iris_model")

    # Predict on the test set, compute and log the loss metric
    y_pred = RFC.predict(test_X)
    accuracy = accuracy_score(test_y, y_pred)
    mlflow.log_metric("accuracy", accuracy)

    print(f"Accuracy: {accuracy}")

2026/04/07 16:41:26 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/07 16:41:26 INFO mlflow.store.db.utils: Updating database tables
2026/04/07 16:41:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/07 16:41:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.9333333333333333


In [9]:
# Load the model back for predictions as a generic Python Function model
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

predictions = loaded_model.predict(test_X)

iris_feature_names = datasets.load_iris().feature_names

result = pd.DataFrame(test_X, columns=iris_feature_names)
result["actual_class"] = test_y
result["predicted_class"] = predictions

result[:4]

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),actual_class,predicted_class
74,NaN,NaN,NaN,NaN,Iris-versicolor,Iris-versicolor
18,NaN,NaN,NaN,NaN,Iris-setosa,Iris-setosa
51,NaN,NaN,NaN,NaN,Iris-versicolor,Iris-versicolor
26,NaN,NaN,NaN,NaN,Iris-setosa,Iris-setosa


In [10]:
joblib.dump(RFC, "model.pkl")

['model.pkl']